# Inference — write `submission.csv`

This notebook produces the **final file you upload to HackerEarth**. It depends only on:

- `predictions/final_test_predictions.parquet` (from `model_blend.ipynb`)
- `e88186124ec611f1 (1)/dataset/test.csv` (for row ordering)
- `e88186124ec611f1 (1)/dataset/sample_submission.csv` (format reference)

It deliberately does **no model work**. All it does is align, clip, and write the CSV with the exact schema HackerEarth expects.


## 1. Setup

In [1]:
import os, json
import numpy as np
import pandas as pd

BASE = '.'
DATASET = os.path.join(BASE, 'e88186124ec611f1 (1)', 'dataset')
PRED_DIR = os.path.join(BASE, 'predictions')

final  = pd.read_parquet(os.path.join(PRED_DIR, 'final_test_predictions.parquet'))
raw_test = pd.read_csv(os.path.join(DATASET, 'test.csv'))
sample = pd.read_csv(os.path.join(DATASET, 'sample_submission.csv'))

print('final_test_predictions:', final.shape, ' cols:', list(final.columns))
print('raw test.csv          :', raw_test.shape)
print('sample_submission.csv :', sample.shape, ' cols:', list(sample.columns))
print('\nsample_submission.csv head:'); print(sample.head())

final_test_predictions: (41778, 2)  cols: ['Index', 'final_demand']
raw test.csv          : (41778, 10)
sample_submission.csv : (5, 2)  cols: ['Index', 'demand']

sample_submission.csv head:
   Index    demand
0      0  0.090768
1      1  0.089885
2      2  0.007037
3      3  0.079087
4      4  0.054636


## 2. Build the submission frame

The expected columns are `Index, demand`. We sort by `Index` ascending to match `test.csv` exactly. We **clip** predictions to `[0, 1.0]` since `demand` is bounded by construction and the metric is R² (so out-of-range predictions only hurt).

In [2]:
# Build the submission ensuring exact row order matches raw test.csv
sub = raw_test[['Index']].merge(final, on='Index', how='left')
sub = sub.rename(columns={'final_demand': 'demand'})

# Sanity
assert len(sub) == len(raw_test) == 41778, f'wrong row count: {len(sub)}'
assert sub['demand'].isna().sum() == 0, 'NaN in predictions'
assert (sub['Index'].values == raw_test['Index'].values).all(), 'Index alignment broken'

# Clip — small safety net
sub['demand'] = sub['demand'].clip(lower=1e-7, upper=1.0)
print(f'Submission row count: {len(sub)}')
print(f'Demand stats: min={sub["demand"].min():.5f}, max={sub["demand"].max():.5f}, '
      f'mean={sub["demand"].mean():.5f}, std={sub["demand"].std():.5f}')

Submission row count: 41778
Demand stats: min=0.00354, max=0.99965, mean=0.08029, std=0.11652


## 3. Distribution sanity by RoadType_modal

A healthy submission should mirror the train target's RoadType-level differential (Highway ≈ 0.6, Street ≈ 0.27, Residential ≈ 0.057). If our means are way off, something is wrong upstream.

In [3]:
test_features = pd.read_parquet(os.path.join(BASE, 'processed', 'test_features.parquet'))
joined = sub.merge(test_features[['Index','RoadType_modal']], on='Index', how='left')
print('Mean prediction by RoadType_modal:')
print(joined.groupby('RoadType_modal')['demand'].agg(['mean','median','count','std']))

Mean prediction by RoadType_modal:
                    mean    median  count       std
RoadType_modal                                     
Highway         0.433504  0.432596   1833  0.052024
Residential     0.059471  0.030986  39334  0.082227
Street          0.360564  0.387050    611  0.064989


## 4. Distribution sanity by qhour

In [4]:
joined = sub.merge(test_features[['Index','qhour_of_day']], on='Index', how='left')
print('Mean prediction by qhour_of_day:')
print(joined.groupby('qhour_of_day')['demand'].mean())

Mean prediction by qhour_of_day:
qhour_of_day
9     0.112897
10    0.083975
11    0.080509
12    0.080772
13    0.075610
14    0.075033
15    0.078352
16    0.079184
17    0.075550
18    0.076162
19    0.076313
20    0.078175
21    0.074278
22    0.076455
23    0.075917
24    0.076642
25    0.078277
26    0.078263
27    0.077275
28    0.078573
29    0.077192
30    0.077808
31    0.079494
32    0.080317
33    0.080442
34    0.077708
35    0.078262
36    0.080760
37    0.081417
38    0.078595
39    0.080380
40    0.082851
41    0.085961
42    0.085064
43    0.085834
44    0.085832
45    0.086430
46    0.082985
47    0.084585
48    0.082931
49    0.082819
50    0.080008
51    0.079193
52    0.079980
53    0.078094
54    0.075908
55    0.074838
Name: demand, dtype: float64


## 5. Write submission.csv

In [5]:
out_path = os.path.join(BASE, 'submission.csv')
sub[['Index','demand']].to_csv(out_path, index=False)
print(f'Wrote {out_path}  ({os.path.getsize(out_path)} bytes)')

# Read back to verify
sub_back = pd.read_csv(out_path)
assert sub_back.shape == (41778, 2)
assert list(sub_back.columns) == ['Index','demand']
assert sub_back['demand'].dtype in (np.float64, np.float32)
print('Round-trip ✓  →  submission.csv is ready to upload.')

Wrote .\submission.csv  (1130504 bytes)
Round-trip ✓  →  submission.csv is ready to upload.
